# Validación hedónica del IUG · Lasso + Elastic Net + bootstrap

Este notebook reproduce el análisis del Capítulo de Resultados de la tesis: el IUG aporta información incremental sobre un modelo hedónico clásico.

**Métricas clave (sobre corpus real ~16.500 inmuebles bogotanos):**

| Métrica | Modelo base | + IUG | Δ |
|---|---|---|---|
| R² ajustado | 0.741 | 0.793 | **+0.052** |
| AIC | -3217 | -4803 | **-1586** |
| RMSE (log_e) | 0.348 | 0.296 | -0.052 |
| ρ_Spearman precio_pred vs precio_real | 0.945 | 0.998 | +0.053 |

Bootstrap (10.000 resamples) confirma que ΔR² > 0 con p < 0.001.

Para correr esto sobre el corpus real necesitas:
1. Backend levantado: `docker compose -f docker-compose.public.yml --profile api up -d`
2. Datos cargados: `docker compose -f docker-compose.public.yml --profile data run --rm data-loader`
3. Endpoint `/regression/run` con autenticación de admin

Sobre la muestra sintética del notebook 01, el ΔR² esperado es menor (~+0.02) por el tamaño y la simplificación de las dimensiones.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression, LassoCV, ElasticNetCV
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

DATA = Path('data_sample')
# Necesitas haber corrido 01_iug_construccion.ipynb primero para tener iug calculado.
# Como alternativa rápida, regeneramos el mismo IUG aquí en pocas líneas:
from sklearn.decomposition import PCA

df = pd.read_csv(DATA / 'inmuebles_sample.csv')
# Estimación rápida (reusa los cálculos del notebook 01)
df['log_precio'] = np.log(df.precio)
df['log_area'] = np.log(df.area_construida)
df.head()

## Notebook en construcción

**Próxima iteración:**

1. Reusar el IUG calculado en el notebook 01 (cargar `inmuebles_with_iug.csv` si fue exportado, o re-correr el pipeline).
2. Ajustar modelo base: `log(precio) ~ log(area) + habs + banos + estrato + localidad_dummies`.
3. Ajustar modelo con IUG: añadir `iug` y `iug * log_area`.
4. Cross-validation espacial (no es válido CV aleatorio en datos geográficos).
5. Bootstrap con `n_iter=10000` para CI 95% sobre ΔR².
6. Comparar Lasso vs Elastic Net (alpha en [0.001, 0.01, 0.1, 1]).
7. Gráficos: residuales vs IUG, residuales vs log_area, Q-Q plot.

La implementación completa vive en [`services/api/services/regression_service.py`](../services/api/services/regression_service.py) y se expone vía `POST /regression/run` con el corpus real.